In [1]:
from google.colab import files
uploaded = files.upload()


Saving country_wise_latest.csv to country_wise_latest.csv
Saving covid_19_clean_complete.csv to covid_19_clean_complete.csv
Saving day_wise.csv to day_wise.csv
Saving worldometer_data.csv to worldometer_data.csv


In [3]:
!pip install plotly

import plotly.express as px


In [17]:
# Load raw data
raw_df = pd.read_csv('day_wise.csv')

# Show first few rows
raw_df.head()


,Date,Confirmed,Deaths,Recovered,Active,New cases,New deaths,New recovered,Deaths / 100 Cases,Recovered / 100 Cases,Deaths / 100 Recovered,No. of countries
0,2020-01-22,555,17,28,510,0,0,0,3.06,5.05,60.71,6
1,2020-01-23,654,18,30,606,99,1,2,2.75,4.59,60.00,8
2,2020-01-24,941,26,36,879,287,8,6,2.76,3.83,72.22,9
3,2020-01-25,1434,42,39,1353,493,16,3,2.93,2.72,107.69,11
4,2020-01-26,2118,56,52,2010,684,14,13,2.64,2.46,107.69,13


In [18]:
raw_df.isnull().sum()


,0
Date,0
Confirmed,0
Deaths,0
Recovered,0
Active,0
New cases,0
New deaths,0
New recovered,0
Deaths / 100 Cases,0
Recovered / 100 Cases,0


In [19]:
print(raw_df.dtypes)

# Convert date column
raw_df['Date'] = pd.to_datetime(raw_df['Date'])


Date                       object
Confirmed                   int64
Deaths                      int64
Recovered                   int64
Active                      int64
New cases                   int64
New deaths                  int64
New recovered               int64
Deaths / 100 Cases        float64
Recovered / 100 Cases     float64
Deaths / 100 Recovered    float64
No. of countries            int64
dtype: object


In [7]:
!pip install dash dash-bootstrap-components pyngrok plotly


In [12]:
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output
import dash_bootstrap_components as dbc
from pyngrok import ngrok
import threading

# Load the dataset
df = pd.read_csv('covid_19_clean_complete.csv')
df['Date'] = pd.to_datetime(df['Date'])

# Initialize Dash app
app = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])
server = app.server

# Dropdown options
countries = df['Country/Region'].unique()

# Layout
app.layout = dbc.Container([
    html.H1("COVID-19 Dashboard", className="text-center my-4"),
    dcc.Dropdown(
        id='country-dropdown',
        options=[{'label': c, 'value': c} for c in countries],
        value='India',
        clearable=False
    ),
    dcc.Graph(id='country-graph')
], fluid=True)

# Callback for updating graph
@app.callback(
    Output('country-graph', 'figure'),
    Input('country-dropdown', 'value')
)
def update_graph(country):
    filtered = df[df['Country/Region'] == country]
    grouped = filtered.groupby('Date')[['Confirmed', 'Deaths', 'Recovered', 'Active']].sum().reset_index()
    melted = grouped.melt(id_vars='Date', value_vars=['Confirmed', 'Deaths', 'Recovered', 'Active'],
                          var_name='Status', value_name='Count')
    fig = px.line(melted, x='Date', y='Count', color='Status',
                  title=f'COVID-19 Trend in {country}')
    return fig

# Run the Dash app in a separate thread
def run_dash():
    app.run_server(port=8050)

thread = threading.Thread(target=run_dash)
thread.start()

# Expose the port using ngrok
public_url = ngrok.connect(8050)
print(f"🔗 Dashboard is live here: {public_url}")


Exception in thread Thread-9 (run_dash):
Traceback (most recent call last):
  File "/usr/lib/python3.11/threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.11/threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "<ipython-input-12-18e79423d0d6>", line 47, in run_dash
  File "/usr/local/lib/python3.11/dist-packages/dash/_obsolete.py", line 22, in __getattr__
    raise err.exc(err.message)
dash.exceptions.ObsoleteAttributeException: app.run_server has been replaced by app.run


🔗 Dashboard is live here: NgrokTunnel: "https://ddf6-34-106-237-209.ngrok-free.app" -> "http://localhost:8050"


In [14]:
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output
import dash_bootstrap_components as dbc
from pyngrok import ngrok
import threading, time

# Load datasets
day_df = pd.read_csv('day_wise.csv')
country_df = pd.read_csv('country_wise_latest.csv')
world_df = pd.read_csv('worldometer_data.csv')

# Convert date column
day_df['Date'] = pd.to_datetime(day_df['Date'])

# Create app
app = Dash(__name__, external_stylesheets=[dbc.themes.LUX])
server = app.server

app.layout = dbc.Container([
    html.H2("COVID-19 Interactive Dashboard", className="text-center my-3"),

    dcc.Tabs([
        dcc.Tab(label='📈 Day-wise Trend', children=[
            dcc.Graph(
                figure=px.line(
                    day_df, x='Date', y=['Confirmed', 'Deaths', 'Recovered', 'Active'],
                    title="Global Day-wise COVID-19 Trends"
                )
            )
        ]),

        dcc.Tab(label='🌍 Country Stats', children=[
            html.Br(),
            dcc.Dropdown(
                id='country-dropdown',
                options=[{'label': c, 'value': c} for c in country_df['Country/Region'].unique()],
                value='India',
                clearable=False
            ),
            dcc.Graph(id='country-line')
        ]),

        dcc.Tab(label='🧮 Worldometer Summary', children=[
            html.Br(),
            dcc.Graph(
                figure=px.bar(
                    world_df.sort_values('TotalCases', ascending=False).head(10),
                    x='Country/Region', y='TotalCases', color='Country/Region',
                    title='Top 10 Countries by Total Cases'
                )
            ),
            html.Br(),
            dcc.Graph(
                figure=px.pie(
                    world_df,
                    values='TotalDeaths', names='Country/Region',
                    title='COVID-19 Death Distribution by Country'
                )
            )
        ])
    ])
], fluid=True)

# Callback for Country Tab
@app.callback(
    Output('country-line', 'figure'),
    Input('country-dropdown', 'value')
)
def update_country(country):
    # Reuse country_df for this since it's already grouped
    row = country_df[country_df['Country/Region'] == country].squeeze()
    data = {
        'Metric': ['Confirmed', 'Deaths', 'Recovered', 'Active'],
        'Count': [row['Confirmed'], row['Deaths'], row['Recovered'], row['Active']]
    }
    df = pd.DataFrame(data)
    fig = px.bar(df, x='Metric', y='Count', color='Metric', title=f'Current Stats for {country}')
    return fig

# Run server in background thread
def run_dash():
    app.run(host='0.0.0.0', port=8050)

thread = threading.Thread(target=run_dash)
thread.start()

time.sleep(3)  # wait for server
public_url = ngrok.connect(8050)
print(f"🔗 Dashboard is live: {public_url}")


<IPython.core.display.Javascript object>

🔗 Dashboard is live: NgrokTunnel: "https://8fc2-34-106-237-209.ngrok-free.app" -> "http://localhost:8050"


In [11]:
from pyngrok import ngrok

# Replace with your actual ngrok authtoken
ngrok.set_auth_token("2wRSQuRcUhBQRE4t1x6PSaUp3Qh_5rWaQpSqjNAEDTcYwRn2k")


In [2]:
import pandas as pd

# Load all CSVs
latest_df = pd.read_csv('country_wise_latest.csv')
full_df = pd.read_csv('covid_19_clean_complete.csv')
daywise_df = pd.read_csv('day_wise.csv')
worldometer_df = pd.read_csv('worldometer_data.csv')

# Show the first few rows of each
print("Latest Data:\n", latest_df.head(), "\n")
print("Full Data:\n", full_df.head(), "\n")
print("Day-wise Data:\n", daywise_df.head(), "\n")
print("Worldometer Data:\n", worldometer_df.head(), "\n")


Latest Data:
   Country/Region  Confirmed  Deaths  Recovered  Active  New cases  New deaths  \
0    Afghanistan      36263    1269      25198    9796        106          10   
1        Albania       4880     144       2745    1991        117           6   
2        Algeria      27973    1163      18837    7973        616           8   
3        Andorra        907      52        803      52         10           0   
4         Angola        950      41        242     667         18           1   

   New recovered  Deaths / 100 Cases  Recovered / 100 Cases  \
0             18                3.50                  69.49   
1             63                2.95                  56.25   
2            749                4.16                  67.34   
3              0                5.73                  88.53   
4              0                4.32                  25.47   

   Deaths / 100 Recovered  Confirmed last week  1 week change  \
0                    5.04                35526         